## Now we need our Language Encoder to contrast to our SiglipVisionModel

In [1]:
IMAGENET_STANDARD_MEAN = [0.5, 0.5, 0.5]
IMAGENET_STANDARD_STD = [0.5, 0.5, 0.5]

In [3]:
from typing import Dict, List, Optional, Union, Tuple, Iterable
from PIL import Image
class PaliGemmaProcessor:
    "givne text and an image, create text token with placeholder for the image. "
    IMAGE_TOKEN = "<image>"

    def __init__(self,tokenizer, num_image_tokens: int, image_size: int):
        super().__init__()

        self.image_seq_length = num_image_tokens
        self.image_size = num_image_tokens

        #Tokeniser described here: https://github.com/google-research/big_vision/blob/main/big_vision/configs/proj/paligemma/README.md#tokenizer
        #We further extend its vocabulary with 1024 entries that represent coordinates in normalized image-space (<loc0000>...<loc1023>), and another with 128 entries (<seg000>...<seg127>) 
        # that are codewords used by a lightweight referring-expression segmentation vector-quantized variational auto-encoder (VQ-VAE) 
        #PaliGemma can do segmentation, detection
        # also, https://huggingface.co/blog/paligemma
        tokens_to_add = {'additional_special_tokens': self.IMAGE_TOKEN}
        tokenizer.add_special_tokens(tokens_to_add)
        EXTRA_TOKENS = [
            f"<loc{i:04d}>" for i in range(1024)
        ]  # These tokens are used for object detection (bounding boxes)
        EXTRA_TOKENS += [
            f"<seg{i:03d}>" for i in range(128)
        ]  # These tokens are used for object segmentation
        tokenizer.add_tokens(EXTRA_TOKENS)
        self.image_token_id = tokenizer.convert_tokens_to_ids(self.IMAGE_TOKEN)
        # We will add the BOS and EOS tokens ourselves
        tokenizer.add_bos_token = False
        tokenizer.add_eos_token = False

        self.tokenizer = tokenizer

    def __call__(
            self,
            text: List[str],
            images: List[Image.Image]) -> dict:
        assert len(images) == 1 and len(text) == 1, f"Received {len(images)} images for {len(text)} prompts."